# Paper 2 · Fix experiment (censoring-aware training) + ConvNeXt repeat of V2

**Before running, in the right-hand panel:**
1. **Accelerator:** GPU T4 x2
2. **Internet:** ON
3. **Inputs (Add Input), both:** the `derm7pt` dataset (as in the V2 run) **and** `skin-cancer-mnist-ham10000` (by kmader)

Then **Save Version → Save & Run All (Commit)**. Expected: about 2.5–3 hours (estimate).

- GPU 0: 6 Derm7pt EfficientNet-B4 models with the censoring-aware loss.
- GPU 1: 6 Derm7pt ConvNeXt-Tiny models (repeat of V2), then 6 Derma-XAI concept models with the masked loss.

**Download:** `p2_fix.zip` from the Output tab. The `weights/` folder can stay in Kaggle.


In [ ]:
# Cell 1 (public version): copy the D1 label files built by tables/step1_build_tables.py.
# Upload train_labels.csv and test_images.csv as a private Kaggle dataset and add it as an input.
# The MD5 check confirms they match the files used in the paper.
import glob, hashlib, os, shutil
os.makedirs('/kaggle/working/d1_labels', exist_ok=True)
for name, want in [('train_labels.csv', '5c4002fbf598'), ('test_images.csv', 'd91f29aee05c')]:
    src = glob.glob('/kaggle/input/**/' + name, recursive=True)
    assert src, name + ' not found: add your label dataset as a notebook input'
    data = open(src[0], 'rb').read()
    got = hashlib.md5(data).hexdigest()[:12]
    assert got == want, (name, got, want)
    open('/kaggle/working/d1_labels/' + name, 'wb').write(data)
    print(name, 'OK', got)


In [ ]:
%%writefile /kaggle/working/train_fix_derm7pt.py
# Paper 2 - FIX experiment, Derm7pt part (pre-registered in the D1 protocol,
# "Pre-registration of the fix"). Built from the V2 script; adds LOSS = "aware"
# (censoring-aware partial-label loss) next to LOSS = "std" (unchanged V2 loss).
# Earlier description of V2 follows. Same recipe as the EASY replication job, except
# that the Derm7pt CONCEPT labels used for training are censored with
# probability PI: on naevi, melanoma-side categories are set to "absent";
# on melanomas, naevus-side categories are set to "absent". Diagnosis labels
# and all other lesions are untouched.
# 1) Train the concept encoder on ALL 1011 Derm7pt images (censored labels).
# 2) Save the weights and the censored label table.
# 3) Predict concept probabilities + black-box melanoma probability on EASY.
# Settings come from environment variables set by cell 2:
#   P2_JOBS      e.g. "effb4:0:1:aware,convnext_tiny:0:0:std"  (backbone:seed:pi:loss)
#   P2_EASY_DIR  folder with the EASY ISIC_*.jpg images
#   P2_INPUT     Kaggle input root (default /kaggle/input)
#   P2_OUT       output folder (default /kaggle/working)
#   P2_SMOKE=1   tiny local test only
import os
import glob
import time
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T
import torchvision.models as tvm
from sklearn.metrics import roc_auc_score

SMOKE = os.environ.get("P2_SMOKE", "0") == "1"
INPUT_ROOT = os.environ.get("P2_INPUT", "/kaggle/input")
OUT_DIR = os.environ.get("P2_OUT", "/kaggle/working")
EASY_DIR = os.environ.get("P2_EASY_DIR", "/tmp/easy_images")
JOB_TEXT = os.environ.get("P2_JOBS", "effb4:0:0")
JOBS = []
for j in JOB_TEXT.split(","):
    if j:
        name, sd, pi, lossname = j.split(":")
        assert lossname in ("std", "aware")
        JOBS.append((name, int(sd), float(pi), lossname))
EPOCHS = 2 if SMOKE else 40
# identical to the confirmation runs (kaggle_dump_concepts.py)
RECIPE = {
    "effb4": dict(IMG=380, BS=16, LR=3e-4, WD=1e-4),
    "convnext_tiny": dict(IMG=384, BS=16, LR=1e-4, WD=5e-2),
}
CONCEPTS = {
    "pigment_network": ["absent", "typical", "atypical"],
    "blue_whitish_veil": ["absent", "present"],
    "vascular_structures": [
        "absent", "arborizing", "comma", "hairpin",
        "within regression", "wreath", "dotted", "linear irregular"],
    "streaks": ["absent", "regular", "irregular"],
    "pigmentation": [
        "absent", "diffuse regular", "localized regular",
        "diffuse irregular", "localized irregular"],
    "dots_and_globules": ["absent", "regular", "irregular"],
    "regression_structures": [
        "absent", "blue areas", "white areas", "combinations"],
}
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = DEVICE == "cuda"
if not SMOKE and DEVICE != "cuda":
    raise SystemExit("No GPU visible to this job.")
os.makedirs(os.path.join(OUT_DIR, "weights"), exist_ok=True)


def key_of(path):
    parts = path.replace("\\", "/").split("/")
    return "/".join(parts[-2:]).lower()


# ---------------- Derm7pt (training data) ----------------
pattern = os.path.join(INPUT_ROOT, "**", "meta.csv")
meta = None
for mp in sorted(glob.glob(pattern, recursive=True)):
    d = pd.read_csv(mp)
    if len(d) == 1011 and "derm" in d.columns:
        meta = d
        print("Derm7pt meta.csv:", mp, flush=True)
        break
if meta is None:
    raise SystemExit("Derm7pt meta.csv (1011 rows) not found under "
                     + INPUT_ROOT + " -> add the derm7pt dataset")
index = {}
all_files = glob.glob(os.path.join(INPUT_ROOT, "**", "*"), recursive=True)
for p in all_files:
    if p.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")):
        index[key_of(p)] = p
paths = [index.get(key_of(d)) for d in meta["derm"]]
missing = [d for d, p in zip(meta["derm"], paths) if p is None]
if missing:
    raise SystemExit(str(len(missing)) + " Derm7pt images missing")
meta["img_path"] = paths
meta["mel"] = meta["diagnosis"].str.contains("melanoma").astype(int)
for c, cats in CONCEPTS.items():
    bad = set(meta[c]) - set(cats)
    assert not bad, "unexpected categories in " + c
    lookup = {k: i for i, k in enumerate(cats)}
    meta[c + "__y"] = meta[c].map(lookup)
n_mel = int(meta.mel.sum())
print("Derm7pt:", len(meta), "images,", n_mel, "melanomas", flush=True)

MEL_SIDE = {"pigment_network": ["atypical"], "streaks": ["irregular"],
            "pigmentation": ["diffuse irregular", "localized irregular"],
            "dots_and_globules": ["irregular"],
            "regression_structures": ["blue areas", "white areas",
                                      "combinations"],
            "blue_whitish_veil": ["present"],
            "vascular_structures": ["dotted", "linear irregular"]}
NEV_SIDE = {"pigment_network": ["typical"], "streaks": ["regular"],
            "pigmentation": ["diffuse regular", "localized regular"],
            "dots_and_globules": ["regular"]}


def censor(meta, pi):
    """Censored copy of the Derm7pt labels. One uniform draw per
    (image, criterion), seed fixed by pi, identical for every seed."""
    m = meta.copy()
    rng = np.random.default_rng(2026 + int(round(pi * 100)))
    dx = m["diagnosis"].str.lower()
    is_mel = dx.str.contains("melanoma").values
    is_nev = (dx.str.contains("nevus") & ~dx.str.contains("melanoma")).values
    changed = 0
    for c, cats in CONCEPTS.items():
        u = rng.random(len(m))
        drop = np.zeros(len(m), bool)
        if c in MEL_SIDE:
            drop |= is_nev & m[c].isin(MEL_SIDE[c]).values
        if c in NEV_SIDE:
            drop |= is_mel & m[c].isin(NEV_SIDE[c]).values
        drop &= u < pi
        changed += int(drop.sum())
        m.loc[drop, c] = "absent"
        lookup = {k: i for i, k in enumerate(cats)}
        m[c + "__y"] = m[c].map(lookup)
    return m, changed


def candidate_masks(m, aware):
    """Per image, per criterion: 0/1 mask over its categories of the labels the loss
    accepts. std: only the recorded category. aware: when a naevus records "absent", the
    melanoma-side categories of that criterion are also accepted (censoring could have
    hidden them); likewise naevus-side categories for a melanoma recording "absent"."""
    dx = m["diagnosis"].str.lower()
    is_mel = dx.str.contains("melanoma").values
    is_nev = (dx.str.contains("nevus") & ~dx.str.contains("melanoma")).values
    out = []
    for c, cats in CONCEPTS.items():
        M = np.zeros((len(m), len(cats)), np.float32)
        M[np.arange(len(m)), m[c + "__y"].values] = 1
        if aware:
            absent = (m[c].values == "absent")
            for side, who in ((MEL_SIDE, is_nev), (NEV_SIDE, is_mel)):
                if c in side:
                    for cat in side[c]:
                        M[absent & who, cats.index(cat)] = 1
        out.append(M)
    return np.concatenate(out, 1)

# ---------------- EASY (external test images) ----------------
pattern = os.path.join(EASY_DIR, "**", "ISIC_*.jp*g")
easy_paths = sorted(glob.glob(pattern, recursive=True))
easy_ids = [os.path.splitext(os.path.basename(p))[0] for p in easy_paths]
if not SMOKE and len(set(easy_ids)) != 248:
    raise SystemExit("expected 248 EASY images, found "
                     + str(len(set(easy_ids))))
easy = pd.DataFrame({"isic_id": easy_ids, "img_path": easy_paths})
easy = easy.drop_duplicates("isic_id")
# Quick gate check only (Benign/Malignant from ISIC). The official
# gate is computed later on CPU from the repository metadata.
easy_labels = None
LABEL_COLS = ["diagnosis_1", "benign_malignant"]
pattern = os.path.join(EASY_DIR, "**", "*.csv")
for mp in glob.glob(pattern, recursive=True):
    md = pd.read_csv(mp)
    md.columns = [c.lower() for c in md.columns]
    cols = [c for c in LABEL_COLS if c in md.columns]
    if "isic_id" in md.columns and cols:
        md = md.drop_duplicates("isic_id").set_index("isic_id")
        lab = md[cols[0]].astype(str).str.lower()
        easy_labels = lab.map({"malignant": 1, "benign": 0})
        break
label_note = "found" if easy_labels is not None else "not found"
print("EASY:", len(easy), "images; labels for gate:", label_note,
      flush=True)

MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]


def make_tf(img, train):
    steps = [T.Resize((img, img))]
    if train:
        steps += [T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
                  T.RandomRotation(20),
                  T.ColorJitter(0.1, 0.1, 0.1, 0.02)]
    steps += [T.ToTensor(), T.Normalize(MEAN, STD)]
    return T.Compose(steps)


class TrainDS(Dataset):
    def __init__(self, df, tf, masks):
        self.df = df.reset_index(drop=True)
        self.tf = tf
        self.masks = masks

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        r = self.df.iloc[i]
        y = torch.tensor([r[c + "__y"] for c in CONCEPTS],
                         dtype=torch.long)
        x = self.tf(Image.open(r.img_path).convert("RGB"))
        return x, y, torch.tensor(float(r.mel)), torch.from_numpy(self.masks[i])


class PredDS(Dataset):
    def __init__(self, df, tf):
        self.df = df.reset_index(drop=True)
        self.tf = tf

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        r = self.df.iloc[i]
        return self.tf(Image.open(r.img_path).convert("RGB")), i


class Net(nn.Module):
    def __init__(self, bb, nf):
        super().__init__()
        self.bb = bb
        self.drop = nn.Dropout(0.3)
        self.heads = nn.ModuleList(
            [nn.Linear(nf, len(v)) for v in CONCEPTS.values()])
        self.mel = nn.Linear(nf, 1)

    def forward(self, x):
        h = self.drop(self.bb(x))
        return [hd(h) for hd in self.heads], self.mel(h).squeeze(1)


def build(backbone):
    try:
        if backbone == "convnext_tiny":
            w = None if SMOKE else tvm.ConvNeXt_Tiny_Weights.IMAGENET1K_V1
            bb = tvm.convnext_tiny(weights=w)
            nf = bb.classifier[2].in_features
            bb.classifier[2] = nn.Identity()
        elif SMOKE:
            bb = tvm.efficientnet_b0(weights=None)
            nf = bb.classifier[1].in_features
            bb.classifier = nn.Identity()
        else:
            w = tvm.EfficientNet_B4_Weights.IMAGENET1K_V1
            bb = tvm.efficientnet_b4(weights=w)
            nf = bb.classifier[1].in_features
            bb.classifier = nn.Identity()
    except Exception as e:
        raise SystemExit("Could not get ImageNet weights: " + str(e)
                         + " -> turn Internet ON")
    return Net(bb, nf).to(DEVICE)


@torch.no_grad()
def predict(model, df, img, bs):
    model.eval()
    rows = []
    loader = DataLoader(PredDS(df, make_tf(img, False)), bs,
                        shuffle=False, num_workers=2)
    for x, idx in loader:
        with torch.autocast(DEVICE, enabled=USE_AMP):
            outs, m = model(x.to(DEVICE))
        probs = [F.softmax(o.float(), 1).cpu().numpy() for o in outs]
        pm = torch.sigmoid(m.float()).cpu().numpy()
        for b in range(len(idx)):
            r = {"isic_id": df.iloc[int(idx[b])].isic_id,
                 "p_mel_bb": float(pm[b])}
            for (c, cats), P in zip(CONCEPTS.items(), probs):
                for k, cat in enumerate(cats):
                    r[c + "__" + cat] = float(P[b, k])
            rows.append(r)
    return pd.DataFrame(rows)


SPLITS = np.cumsum([len(v) for v in CONCEPTS.values()])[:-1]
for backbone, seed, PI, LOSS in JOBS:
    cfg = RECIPE[backbone]
    IMG, BS = (64, 8) if SMOKE else (cfg["IMG"], cfg["BS"])
    tag = "fix_pi%03d_%s_%s_seed%d" % (int(round(PI * 100)), LOSS, backbone, seed)
    pred_csv = os.path.join(OUT_DIR, "easy_preds_" + tag + ".csv")
    tmeta, n_changed = censor(meta, PI)
    print("==", tag, ": censoring pi =", PI, "changed", n_changed,
          "concept labels", flush=True)
    tmeta.drop(columns=["img_path"]).to_csv(
        os.path.join(OUT_DIR, "train_labels_" + tag + ".csv"), index=False)
    masks = candidate_masks(tmeta, LOSS == "aware")
    print("==", tag, ": loss", LOSS, "- labels accepting more than one category:",
          int((masks.sum(1) > len(CONCEPTS)).sum()), "images", flush=True)
    if os.path.exists(pred_csv):
        print("skip (exists):", pred_csv, flush=True)
        continue
    # full-data runs have no fold, so the seed is used directly
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    model = build(backbone)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["LR"],
                            weight_decay=cfg["WD"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS)
    scaler = torch.amp.GradScaler(enabled=USE_AMP)
    dl = DataLoader(TrainDS(tmeta, make_tf(IMG, True), masks), BS,
                    shuffle=True, num_workers=2, drop_last=True)
    print("==", tag, ": training on all", len(tmeta),
          "Derm7pt images,", EPOCHS, "epochs, img", IMG, flush=True)
    log = []
    for ep in range(1, EPOCHS + 1):
        model.train()
        t0 = time.time()
        tot = 0.0
        for x, y, m, mk in dl:
            x, y, m, mk = x.to(DEVICE), y.to(DEVICE), m.to(DEVICE), mk.to(DEVICE)
            with torch.autocast(DEVICE, enabled=USE_AMP):
                outs, pm = model(x)
                # partial-label loss: -log(total probability of the accepted categories).
                # With one accepted category this equals cross-entropy (std = V2 loss).
                ce = []
                for j, (o, mj) in enumerate(zip(outs, torch.tensor_split(mk, SPLITS.tolist(), 1))):
                    lp = F.log_softmax(o.float(), 1)
                    ce.append(-torch.logsumexp(lp + torch.log(mj.clamp_min(1e-12)), 1).mean())
                bce = F.binary_cross_entropy_with_logits(pm, m)
                loss = sum(ce) + bce
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            tot += loss.item()
        sched.step()
        msg = {"epoch": ep, "loss": round(tot / max(len(dl), 1), 4),
               "sec": round(time.time() - t0, 1)}
        log.append(msg)
        if ep in (1, EPOCHS) or ep % 10 == 0:
            print("[" + tag + "]", msg, flush=True)
    state = {}
    for k, v in model.state_dict().items():
        state[k] = v.half() if v.is_floating_point() else v
    torch.save(state, os.path.join(OUT_DIR, "weights", tag + ".pt"))
    P = predict(model, easy, IMG, BS * 2)
    P.insert(0, "seed", seed)
    P.insert(0, "backbone", backbone)
    P.to_csv(pred_csv, index=False)
    log_csv = os.path.join(OUT_DIR, "log_full_" + tag + ".csv")
    pd.DataFrame(log).to_csv(log_csv, index=False)
    if easy_labels is not None:
        y = P.isic_id.map(easy_labels)
        ok = y.notna()
        if ok.sum() > 10 and y[ok].nunique() == 2:
            auc = roc_auc_score(y[ok], P.p_mel_bb[ok])
            print("[" + tag + "] EASY gate: black-box melanoma AUC =",
                  round(auc, 3), "(n =", int(ok.sum()),
                  "; pre-registered gate >= 0.75)", flush=True)
    print("[" + tag + "] wrote", pred_csv, "and weights", flush=True)
    del model
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
print("DONE", flush=True)


In [ ]:
%%writefile /kaggle/working/train_dxai_masked.py
# Paper 2 - FIX experiment, Derma-XAI part (pre-registered in the D1 protocol,
# "Pre-registration of the fix"). Same data, split, recipe and seeds as D1 Step 3 /
# the EASY run, but the CONCEPT model is trained with a censoring-aware masked loss:
# on naevi the melanoma-side characteristics are "unknown" (masked out of the loss), on
# melanomas the naevus-side ones are. Only concept models are trained; the diagnosis
# (black-box) models are reused from the earlier, exactly reproducible run.
# Predictions are written for the 196 test images and the 248 EASY images.
import json
import os
import random
import time
import zipfile

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

LAB = os.environ.get("D1_LABELS", "/kaggle/working/d1_labels")
IMG_ROOT = os.environ.get("D1_IMAGES", "/kaggle/input")
OUT = os.environ.get("D1_OUT", "/kaggle/working/d1_out")
EASY_DIR = os.environ.get("D1_EASY", "/kaggle/working/easy_images")
WDIR = os.environ.get("D1_WEIGHTS_DIR", "/kaggle/working/weights")
os.makedirs(WDIR, exist_ok=True)
SEEDS = [int(s) for s in os.environ.get("D1_SEEDS", "0,1,2,3,4").split(",")]
EPOCHS = int(os.environ.get("D1_EPOCHS", "30"))
SMOKE = os.environ.get("D1_SMOKE", "0") == "1"
WEIGHTS = None if os.environ.get("D1_NOWEIGHTS") == "1" else "IMAGENET1K_V1"
os.makedirs(OUT, exist_ok=True)

MEL = ["TRBL", "ESA", "GP", "PV", "PRL", "BDG", "WLSA", "PLR", "PES", "PIF"]
NEV = ["OPC", "SPC", "MVP", "PRLC", "PLF", "PDES", "APC", "MS"]
ALL = MEL + NEV
dev = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", dev)

train = pd.read_csv(os.path.join(LAB, "train_labels.csv"))
test = pd.read_csv(os.path.join(LAB, "test_images.csv"))
paths = {}
for root, _, files in os.walk(IMG_ROOT, followlinks=True):
    for f in files:
        if f.lower().endswith(".jpg") and f.startswith("ISIC_"):
            paths.setdefault(f[:-4], os.path.join(root, f))
need = set(train.image_id) | set(test.image_id)
missing = sorted(need - set(paths))
print("images found: %d of %d needed" % (len(need) - len(missing), len(need)))
if missing:
    raise SystemExit("missing images, e.g. %s: attach the HAM10000 dataset"
                     % missing[:3])
easy_paths = {}
for root, _, files in os.walk(EASY_DIR):
    for f in files:
        if f.startswith("ISIC_") and f.lower().endswith((".jpg", ".jpeg")):
            easy_paths.setdefault(f.split(".")[0], os.path.join(root, f))
print("EASY images found:", len(easy_paths))
if len(easy_paths) != 248:
    raise SystemExit("expected 248 EASY images: run cell 2 first")
for k, v in easy_paths.items():
    paths.setdefault(k, v)
easy = pd.DataFrame({"image_id": sorted(easy_paths)})
if SMOKE:
    train = pd.concat([train[train.split == "train"].head(40),
                       train[train.split == "val"].head(16)])
    test = test.head(16)

norm = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
T_TRAIN = transforms.Compose([
    transforms.Resize((224, 224)), transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.02),
    transforms.ToTensor(), norm])
T_EVAL = transforms.Compose([transforms.Resize((224, 224)),
                             transforms.ToTensor(), norm])


class DS(Dataset):
    def __init__(self, df, tf, target):
        self.df, self.tf, self.target = df.reset_index(drop=True), tf, target

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        r = self.df.iloc[i]
        x = self.tf(Image.open(paths[r.image_id]).convert("RGB"))
        if self.target == "none":
            return x, torch.zeros(1)
        elif self.target == "concepts":
            y = torch.tensor(r[ALL].values.astype(np.float32))
            mel = float(r.benign_malignant) == 1.0
            mk = torch.tensor([1.0 if (c in MEL) == mel else 0.0 for c in ALL])
            return x, y, mk
        else:
            y = torch.tensor([float(r.benign_malignant)])
        return x, y


def backbone(name, n_out):
    if name == "effb0":
        m = models.efficientnet_b0(weights=WEIGHTS)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, n_out)
    elif name == "convnext_tiny":
        m = models.convnext_tiny(weights=WEIGHTS)
        m.classifier[2] = nn.Linear(m.classifier[2].in_features, n_out)
    else:
        raise ValueError(name)
    return m


def predict(m, df, target):
    dl = DataLoader(DS(df, T_EVAL, "none"), batch_size=64, num_workers=2)
    m.eval()
    out = []
    with torch.no_grad():
        for x, _ in dl:
            out.append(torch.sigmoid(m(x.to(dev))).cpu().numpy())
    return np.concatenate(out)


def fit(name, target, seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    tr, va = train[train.split == "train"], train[train.split == "val"]
    n_out = len(ALL) if target == "concepts" else 1
    m = backbone(name, n_out).to(dev)
    opt = torch.optim.AdamW(m.parameters(), lr=3e-4)
    bce = nn.BCEWithLogitsLoss(reduction="none")

    def lossf(z, y, mk):
        return (bce(z, y) * mk).sum() / mk.sum().clamp_min(1.0)
    dl = DataLoader(DS(tr, T_TRAIN, target), batch_size=32, shuffle=True,
                    num_workers=2, drop_last=True)
    dv = DataLoader(DS(va, T_EVAL, target), batch_size=64, num_workers=2)
    best, best_state, bad = 1e9, None, 0
    for ep in range(EPOCHS):
        m.train()
        for x, y, mk in dl:
            opt.zero_grad()
            loss = lossf(m(x.to(dev)), y.to(dev), mk.to(dev))
            loss.backward(); opt.step()
        m.eval(); vl, n = 0.0, 0
        with torch.no_grad():
            for x, y, mk in dv:
                vl += lossf(m(x.to(dev)), y.to(dev), mk.to(dev)).item() * len(x); n += len(x)
        vl /= n
        print("  %s %s seed %d epoch %d val loss %.4f" % (name, target, seed, ep, vl),
              flush=True)
        if vl < best - 1e-4:
            best, bad = vl, 0
            best_state = {k: v.detach().cpu().clone() for k, v in m.state_dict().items()}
        else:
            bad += 1
            if bad >= 5:
                break
    m.load_state_dict(best_state)
    return m, best


RUNS = [("effb0", s) for s in SEEDS] + [("convnext_tiny", SEEDS[0])]
if os.environ.get("D1_RUNS"):
    RUNS = [(a, int(b)) for a, b in (r.split(":") for r in os.environ["D1_RUNS"].split(","))]
summary = []
for name, seed in RUNS:
    t0 = time.time()
    tag = "masked_%s_seed%d" % (name, seed)
    if os.path.exists(os.path.join(OUT, "easy_%s.csv" % tag)):
        print("skip (exists):", tag, flush=True)
        continue
    print("==", tag, ": masked concept training", flush=True)
    mc, lc = fit(name, "concepts", seed)
    pd.DataFrame(predict(mc, test, "concepts"), columns=ALL).assign(
        image_id=test.image_id.values, y=test.benign_malignant.values).to_csv(
        os.path.join(OUT, "test_%s.csv" % tag), index=False)
    pd.DataFrame(predict(mc, easy, "concepts"), columns=ALL).assign(
        image_id=easy.image_id.values).to_csv(os.path.join(OUT, "easy_%s.csv" % tag), index=False)
    torch.save(mc.state_dict(), os.path.join(WDIR, "concepts_%s.pt" % tag))
    summary.append({"run": tag, "val_masked_concept_loss": lc, "minutes": (time.time() - t0) / 60})
    print("[" + tag + "] wrote predictions", summary[-1], flush=True)
pd.DataFrame(summary).to_csv(os.path.join(OUT, "run_summary_masked.csv"), index=False)
print("DONE", flush=True)


In [ ]:
# Cell 4: checks, EASY download, then both parts of the fix experiment on 2 GPUs.
#   GPU 0: Derm7pt EfficientNet-B4 with the censoring-aware loss (pi = 1 first, then pi = 0)
#   GPU 1: Derm7pt ConvNeXt-Tiny, standard loss (V2 repeat), then Derma-XAI masked models
import glob, hashlib, os, subprocess, sys, time, zipfile
try:
    LOCAL_TEST
except NameError:
    LOCAL_TEST = False
WORK = os.environ.get("FIX_WORK", "/kaggle/working")
EASY = WORK + "/easy_images"
OUTD = WORK + "/fix_out"
os.makedirs(OUTD + "/dxai", exist_ok=True)
SCRIPTS = {"train_fix_derm7pt.py": "b102e928e199", "train_dxai_masked.py": "de6f11ca188a"}
for name, want in SCRIPTS.items():
    p = WORK + "/" + name
    assert os.path.exists(p), "Run the cell that writes " + name + " first"
    ok = subprocess.run([sys.executable, "-m", "py_compile", p]).returncode == 0
    fp = hashlib.md5("".join(open(p).read().split()).encode()).hexdigest()[:12]
    print(name, "compiles:", ok, "| fingerprint", fp, "| expected", want)
    if not ok or fp != want:
        raise RuntimeError(name + " is damaged or not the tested version: import paper2_fix.ipynb, do not paste")
if LOCAL_TEST:
    gpus = ["GPU 0 (test)", "GPU 1 (test)"]
else:
    gpus = [l for l in subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.splitlines() if l.strip()]
print("\n".join(gpus) or "NO GPU FOUND")
assert gpus, "Turn on GPU: Settings -> Accelerator -> GPU T4 x2"

def n_images():
    return len({os.path.basename(p).split(".")[0] for p in glob.glob(EASY + "/**/ISIC_*.jp*g", recursive=True)})
os.makedirs(EASY, exist_ok=True)
if not LOCAL_TEST and n_images() != 248:
    import requests   # direct ISIC API download (faster and more reliable than isic-cli)
    url = "https://api.isic-archive.com/api/v2/images/search/?collections=166&limit=100"
    while url:
        page = requests.get(url, timeout=60).json()
        for res in page["results"]:
            p = os.path.join(EASY, res["isic_id"] + ".jpg")
            if not os.path.exists(p):
                open(p, "wb").write(requests.get(res["files"]["full"]["url"], timeout=120).content)
        url = page.get("next")
print("EASY images:", n_images(), flush=True)
assert n_images() == 248, "EASY download incomplete: check Internet is ON, then run this cell again"

GPU0 = "effb4:0:1:aware,effb4:1:1:aware,effb4:2:1:aware,effb4:0:0:aware,effb4:1:0:aware,effb4:2:0:aware"
GPU1 = ("convnext_tiny:0:0:std,convnext_tiny:0:1:std,convnext_tiny:1:0:std,"
        "convnext_tiny:1:1:std,convnext_tiny:2:0:std,convnext_tiny:2:1:std")
DXAI_RUNS = "effb0:0,effb0:1,effb0:2,effb0:3,effb0:4,convnext_tiny:0"
if LOCAL_TEST:
    GPU0, GPU1, DXAI_RUNS = "effb4:0:1:aware", "convnext_tiny:0:1:std", "effb0:0"
base = dict(os.environ, P2_OUT=OUTD, P2_EASY_DIR=EASY, D1_EASY=EASY, D1_OUT=OUTD + "/dxai",
            D1_WEIGHTS_DIR=WORK + "/weights", D1_LABELS=WORK + "/d1_labels")
py = sys.executable
lists = [(0, f'"{py}" -u {WORK}/train_fix_derm7pt.py', dict(P2_JOBS=GPU0)),
         (1, f'"{py}" -u {WORK}/train_fix_derm7pt.py; "{py}" -u {WORK}/train_dxai_masked.py',
          dict(P2_JOBS=GPU1, D1_RUNS=DXAI_RUNS))]
jobs = []
for k, cmd, extra in lists:
    env = dict(base, **extra)
    if not LOCAL_TEST:
        env["CUDA_VISIBLE_DEVICES"] = str(k if len(gpus) >= 2 else 0)
    logf = OUTD + f"/stdout_fix_gpu{k}.txt"
    p = subprocess.Popen(["bash", "-c", cmd], stdout=open(logf, "w"), stderr=subprocess.STDOUT, env=env)
    jobs.append((k, p, logf))
    if len(gpus) < 2 and not LOCAL_TEST:
        p.wait()
KEYS = ("==", "'epoch': 40", "wrote", "DONE", "Error", "Traceback", "skip")
seen = {k: 0 for k, _, _ in jobs}
while True:
    for k, p, logf in jobs:
        lines = open(logf).read().splitlines()
        for line in lines[seen[k]:]:
            if any(s in line for s in KEYS):
                print(f"[gpu{k}]", line, flush=True)
        seen[k] = len(lines)
    if all(p.poll() is not None for _, p, _ in jobs):
        break
    time.sleep(5 if LOCAL_TEST else 60)
files = sorted(f for f in glob.glob(OUTD + "/**/*", recursive=True) if os.path.isfile(f))
with zipfile.ZipFile(WORK + "/p2_fix.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in files:
        z.write(f, os.path.relpath(f, WORK))
n7 = len(glob.glob(OUTD + "/easy_preds_fix_*.csv"))
nx = len(glob.glob(OUTD + "/dxai/easy_masked_*.csv"))
print(f"p2_fix.zip: {len(files)} files | Derm7pt models {n7} (expect {len((GPU0 + ',' + GPU1).split(','))}) | "
      f"Derma-XAI masked models {nx} (expect {len(DXAI_RUNS.split(','))})")
